In [ ]:
import subprocess
import sys
from datetime import datetime, timedelta
import os

# Install required packages for MODIS data downloading
packages = ['requests', 'rasterio', 'xarray']
for package in packages:
    try:
        __import__(package)
        print(f"{package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"{package} installed successfully")

import requests
import pandas as pd
import numpy as np

# MOD13A2 is MODIS Vegetation Indices (NDVI/EVI) with 16-day composite
# Date range: December 1, 2000 to May 24, 2016
start_date = datetime(2000, 12, 1)
end_date = datetime(2016, 5, 24)

print(f"MOD13A2 Download Configuration")
print(f"=" * 60)
print(f"Dataset: MOD13A2 (MODIS Vegetation Indices)")
print(f"Date Range: {start_date.date()} to {end_date.date()}")
print(f"Duration: {(end_date - start_date).days} days")

# MODIS tiles covering Uttar Pradesh, Haryana, and Punjab (IGP region)
# The sinusoidal grid tiles for northern India:
tiles = ['h25v05', 'h25v06', 'h26v05', 'h26v06']

print(f"\nMODIS Tiles (covering UP, Haryana, Punjab):")
for tile in tiles:
    print(f"  - {tile}")

# Create data directories
data_dir = "/media/sam/writable/Sam Rice Yield Pred/data"
mod13a2_dir = os.path.join(data_dir, "MOD13A2")
os.makedirs(mod13a2_dir, exist_ok=True)

print(f"\nData Directory: {mod13a2_dir}")
print(f"\n" + "=" * 60)
print("✓ Configuration ready")
print("✓ Ready to proceed with NASA LAADS DAAC downloads")

In [ ]:
import getpass
import json

# Prompt for NASA Earthdata bearer token (used for download authentication)
print("=" * 60)
print("NASA Earthdata Authentication")
print("=" * 60)
bearer_token = getpass.getpass("Enter your NASA Earthdata bearer token: ")

# Headers for downloading (CMR search does not require auth)
headers = {
    "Authorization": f"Bearer {bearer_token}"
}

print(f"\n✓ Token stored securely")
print(f"✓ Ready to query NASA CMR for MOD13A2")

# Use NASA's Common Metadata Repository (CMR) Search API
# CMR is the canonical metadata service for all NASA Earth science data
print(f"\n" + "=" * 60)
print(f"Querying MOD13A2 granules from NASA CMR")
print("=" * 60)

cmr_url = "https://cmr.earthdata.nasa.gov/search/granules.json"
collection_short_name = "MOD13A2"
collection_version = "061"  # Collection 6.1

# Convert dates to ISO 8601 format for CMR
start_str = start_date.strftime('%Y-%m-%dT%H:%M:%SZ')
end_str = end_date.strftime('%Y-%m-%dT23:59:59Z')

# Query CMR for each tile by filtering producer_granule_id (contains tile name)
search_results = {}
total_files = 0

for tile in tiles:
    print(f"\nQuerying tile {tile}...")
    
    all_granules = []
    page_num = 1
    
    while True:
        params = {
            "short_name": collection_short_name,
            "version": collection_version,
            "temporal": "{},{}".format(start_str, end_str),
            "producer_granule_id": "*{}*".format(tile),  # Wildcard match on tile name
            "options[producer_granule_id][pattern]": "true",
            "page_size": 2000,
            "page_num": page_num
        }
        
        try:
            response = requests.get(cmr_url, params=params, timeout=60)
            response.raise_for_status()
            data = response.json()
            
            granules = data.get('feed', {}).get('entry', [])
            if not granules:
                break
            
            all_granules.extend(granules)
            
            # Check if more pages exist
            if len(granules) < 2000:
                break
            page_num += 1
            
        except requests.exceptions.RequestException as e:
            print("  Error querying CMR: {}".format(str(e)))
            break
    
    # Extract download URLs from granules
    granule_info = []
    for g in all_granules:
        title = g.get('title', '')
        # Find the download link (HDF file)
        for link in g.get('links', []):
            href = link.get('href', '')
            if href.endswith('.hdf') and 'http' in href:
                granule_info.append({
                    'filename': href.split('/')[-1],
                    'download_url': href,
                    'title': title
                })
                break
    
    search_results[tile] = {
        'count': len(granule_info),
        'files': granule_info
    }
    total_files += len(granule_info)
    print("  Found {} files".format(len(granule_info)))

print(f"\n" + "=" * 60)
print(f"Summary: Found {total_files} total MOD13A2 granules")
print(f"Tiles: {len(tiles)}")
print(f"Date range: {start_str} to {end_str}")
print("=" * 60)

# Show a sample to verify
if total_files > 0:
    sample_tile = next(t for t in tiles if search_results[t]['count'] > 0)
    sample_file = search_results[sample_tile]['files'][0]
    print(f"\nSample file from tile {sample_tile}:")
    print(f"  Filename: {sample_file['filename']}")
    print(f"  URL: {sample_file['download_url']}")

print(f"\n✓ Ready to proceed with downloading {total_files} files")

In [ ]:
import os

# Download MOD13A2 files for each tile
# Note: NASA Earthdata authentication uses redirects; allow them with the session
print("=" * 60)
print("Downloading MOD13A2 Files from NASA")
print("=" * 60)

downloaded_count = 0
failed_count = 0
skipped_count = 0

# Use a session that handles redirects properly while preserving the auth header
session = requests.Session()
session.headers.update(headers)

for tile, tile_info in search_results.items():
    files = tile_info.get('files', [])
    num_files = len(files)
    
    print("\n[Tile {}] Processing {} files...".format(tile, num_files))
    
    # Create subdirectory for each tile
    tile_dir = os.path.join(mod13a2_dir, tile)
    os.makedirs(tile_dir, exist_ok=True)
    
    for idx, file_info in enumerate(files, 1):
        filename = file_info['filename']
        download_url = file_info['download_url']
        filepath = os.path.join(tile_dir, filename)
        
        # Skip if already downloaded
        if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
            skipped_count += 1
            continue
        
        try:
            # Stream the download
            response = session.get(download_url, timeout=120, stream=True, allow_redirects=True)
            response.raise_for_status()
            
            # Write to file
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            
            file_size_mb = os.path.getsize(filepath) / (1024 * 1024)
            print("  [{}/{}] {} ({:.1f} MB) ✓".format(idx, num_files, filename, file_size_mb))
            downloaded_count += 1
            
        except Exception as e:
            print("  [{}/{}] {} - FAILED: {}".format(idx, num_files, filename, str(e)))
            failed_count += 1
            # Clean up partial download
            if os.path.exists(filepath):
                os.remove(filepath)
            continue

print("\n" + "=" * 60)
print("Download Summary")
print("=" * 60)
print("✓ Successfully downloaded: {} files".format(downloaded_count))
print("✗ Failed: {} files".format(failed_count))
print("⊘ Skipped (already exist): {} files".format(skipped_count))
print("Total processed: {}".format(downloaded_count + failed_count + skipped_count))
print("\nData location: {}".format(mod13a2_dir))
print("=" * 60)

# Verify downloaded files
print("\nVerifying downloaded files...")
for tile in tiles:
    tile_dir = os.path.join(mod13a2_dir, tile)
    if os.path.exists(tile_dir):
        files = [f for f in os.listdir(tile_dir) if f.endswith('.hdf')]
        total_size_gb = sum(os.path.getsize(os.path.join(tile_dir, f)) for f in files) / (1024**3)
        print("  {}: {} HDF files ({:.2f} GB)".format(tile, len(files), total_size_gb))

In [ ]:
# ============================================================
# CSIF Download Configuration
# ============================================================
# CSIF (Contiguous Solar-Induced Chlorophyll Fluorescence) - Zhang et al.
# Hosted at OSF: https://osf.io/8xqy6/
#
# Two product variants needed:
#   1. clear.inst v2 (CSIF_v2/ folder)  -> for Dec 2000 only (all-sky starts 2001)
#   2. all.daily v1  (all-sky/ folder)  -> for 2001-01-01 onward
#
# Both are 4-day composites at 0.05 deg spatial resolution.

import os
from datetime import datetime, timedelta

# OSF project node
OSF_NODE_ID = "8xqy6"
OSF_API_BASE = "https://api.osf.io/v2"

# Folder IDs (discovered via OSF API exploration)
CSIF_FOLDERS = {
    "clear_inst_v2": {
        "name": "CSIF_v2",
        "folder_id": "5c9b7619aae20b0017b090c9",
        "filename_pattern": "OCO2.SIF.clear.inst.{year}{doy:03d}.v2.nc",
    },
    "all_daily": {
        "name": "all-sky",
        "folder_id": "5bd9a1961385910017689662",
        "filename_pattern": "OCO2.SIF.all.daily.{year}{doy:03d}.nc",
    },
}

# Date range (same as MOD13A2)
csif_start_date = datetime(2000, 12, 1)
csif_end_date = datetime(2016, 5, 24)

# Bounding box: UP, Haryana, Punjab + 2 deg buffer on all sides
bbox = {
    "lat_min": 22.0,
    "lat_max": 34.5,
    "lon_min": 72.0,
    "lon_max": 86.5,
}

# Storage directories
csif_dir = "/media/sam/writable/Sam Rice Yield Pred/data/CSIF"
csif_clear_dir = os.path.join(csif_dir, "clear_inst_v2_subset")  # Dec 2000 only
csif_all_dir = os.path.join(csif_dir, "all_daily_subset")        # 2001+
os.makedirs(csif_clear_dir, exist_ok=True)
os.makedirs(csif_all_dir, exist_ok=True)

print("CSIF Download Configuration")
print("=" * 60)
print("Date Range : {} -> {}".format(csif_start_date.date(), csif_end_date.date()))
print("Bounding box (lat,lon): ({}, {}) -> ({}, {})".format(
    bbox["lat_min"], bbox["lon_min"], bbox["lat_max"], bbox["lon_max"]))
print()
print("Hybrid strategy:")
print("  Dec 2000      -> CSIF_v2 (clear.inst v2)")
print("  2001-01-01 -> 2016-05-24 -> all-sky (all.daily v1)")
print()
print("Storage:")
print("  clear.inst v2 subset: {}".format(csif_clear_dir))
print("  all.daily subset    : {}".format(csif_all_dir))


def doy_from_date(dt):
    """Return Julian day-of-year (1-indexed) for a datetime."""
    return (dt - datetime(dt.year, 1, 1)).days + 1


def expected_doys_for_year(year, start_dt, end_dt):
    """
    CSIF 4-day composites use DOY = 1, 5, 9, ... within each year.
    Return the list of DOYs in the given year that fall within
    [start_dt, end_dt].
    """
    doys = []
    for doy in range(1, 366, 4):
        try:
            file_date = datetime(year, 1, 1) + timedelta(days=doy - 1)
        except ValueError:
            continue
        if start_dt <= file_date <= end_dt:
            doys.append(doy)
    return doys


print("\nExpected file count (estimate):")
print("  Dec 2000 (clear.inst v2): {} files".format(
    len(expected_doys_for_year(2000, csif_start_date, csif_end_date))))
all_daily_total = 0
for y in range(2001, 2017):
    all_daily_total += len(expected_doys_for_year(y, csif_start_date, csif_end_date))
print("  2001-2016 (all.daily)   : {} files".format(all_daily_total))


In [ ]:
# ============================================================
# Enumerate CSIF files via OSF API
# ============================================================
# OSF organizes files as: project -> folder -> year subfolder -> NC files
# We must walk the tree to collect file IDs and download URLs.

def osf_list_folder(folder_url):
    """List entries (files/folders) under an OSF storage folder URL, with pagination."""
    entries = []
    next_url = folder_url
    while next_url:
        resp = requests.get(next_url, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        entries.extend(data.get("data", []))
        next_url = data.get("links", {}).get("next")
    return entries


def osf_folder_url(folder_id):
    return "{}/nodes/{}/files/osfstorage/{}/".format(
        OSF_API_BASE, OSF_NODE_ID, folder_id
    )


# Build inventory of files we want to download
csif_inventory = {"clear_inst_v2": [], "all_daily": []}

print("Enumerating CSIF files from OSF...")
print("=" * 60)

for variant_key, variant_info in CSIF_FOLDERS.items():
    print("\nVariant: {} ({})".format(variant_key, variant_info["name"]))
    print("-" * 60)

    # List year subfolders inside this variant's folder
    year_folders = osf_list_folder(osf_folder_url(variant_info["folder_id"]))

    for yf in year_folders:
        if yf["attributes"]["kind"] != "folder":
            continue
        year_name = yf["attributes"]["name"]
        try:
            year = int(year_name)
        except ValueError:
            continue

        # Hybrid strategy filter:
        #   clear_inst_v2 -> ONLY 2000 (Dec 2000)
        #   all_daily     -> 2001 through 2016
        if variant_key == "clear_inst_v2" and year != 2000:
            continue
        if variant_key == "all_daily" and not (2001 <= year <= 2016):
            continue

        # Compute the DOYs we need from this year
        wanted_doys = set(expected_doys_for_year(year, csif_start_date, csif_end_date))
        if not wanted_doys:
            continue

        # List files in this year folder
        year_folder_id = yf["attributes"]["path"].strip("/")
        year_files = osf_list_folder(osf_folder_url(year_folder_id))

        matched = 0
        for ff in year_files:
            if ff["attributes"]["kind"] != "file":
                continue
            fname = ff["attributes"]["name"]
            if not fname.endswith(".nc"):
                continue

            # Parse DOY out of filename: OCO2.SIF.{sky}.{temp}.YYYYDDD[.v2].nc
            m = re.search(r"\.(\d{4})(\d{3})(?:\.v\d+)?\.nc$", fname)
            if not m:
                continue
            file_year = int(m.group(1))
            file_doy = int(m.group(2))
            if file_year != year or file_doy not in wanted_doys:
                continue

            download_url = ff["links"]["download"]
            csif_inventory[variant_key].append({
                "year": file_year,
                "doy": file_doy,
                "filename": fname,
                "download_url": download_url,
            })
            matched += 1

        print("  {}: {} files matched (wanted {})".format(year, matched, len(wanted_doys)))

print("\n" + "=" * 60)
print("Inventory Summary")
print("=" * 60)
for k, v in csif_inventory.items():
    print("  {}: {} files".format(k, len(v)))
total_csif = sum(len(v) for v in csif_inventory.values())
print("  TOTAL: {} files".format(total_csif))

# Show a sample
if csif_inventory["clear_inst_v2"]:
    s = csif_inventory["clear_inst_v2"][0]
    print("\nSample (clear_inst_v2): {} -> {}".format(s["filename"], s["download_url"]))
if csif_inventory["all_daily"]:
    s = csif_inventory["all_daily"][0]
    print("Sample (all_daily)    : {} -> {}".format(s["filename"], s["download_url"]))


In [ ]:
# ============================================================
# Download CSIF files and save spatial subset (UP/Haryana/Punjab + 2deg buffer)
# ============================================================
# Strategy: stream each global NetCDF to a temp file, open with xarray,
# crop to bbox, write subset, then delete the global file.

import tempfile
import xarray as xr

print("Downloading CSIF files and subsetting to bounding box...")
print("=" * 60)
print("bbox: lat [{}, {}], lon [{}, {}]".format(
    bbox["lat_min"], bbox["lat_max"], bbox["lon_min"], bbox["lon_max"]))
print()

csif_session = requests.Session()


def download_and_subset(file_info, output_dir):
    """Download a CSIF NetCDF, write a bbox-cropped subset, return (success, msg)."""
    out_path = os.path.join(output_dir, file_info["filename"])

    # Skip if already subset
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return ("skip", "exists")

    # Download to temp file
    tmp_fd, tmp_path = tempfile.mkstemp(suffix=".nc")
    os.close(tmp_fd)
    try:
        resp = csif_session.get(file_info["download_url"], stream=True, timeout=180)
        resp.raise_for_status()
        with open(tmp_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1 << 16):
                if chunk:
                    f.write(chunk)

        # Open, subset, write
        ds = xr.open_dataset(tmp_path)

        # CSIF NetCDF uses 'lat' and 'lon' coords; latitudes typically descend (90 -> -90)
        lat_name = "lat" if "lat" in ds.coords else "latitude"
        lon_name = "lon" if "lon" in ds.coords else "longitude"

        # Determine ordering of lat axis
        lat_vals = ds[lat_name].values
        if lat_vals[0] > lat_vals[-1]:
            lat_slice = slice(bbox["lat_max"], bbox["lat_min"])
        else:
            lat_slice = slice(bbox["lat_min"], bbox["lat_max"])
        lon_slice = slice(bbox["lon_min"], bbox["lon_max"])

        ds_sub = ds.sel({lat_name: lat_slice, lon_name: lon_slice})

        # Write with NetCDF4 + compression
        encoding = {var: {"zlib": True, "complevel": 4}
                    for var in ds_sub.data_vars}
        ds_sub.to_netcdf(out_path, encoding=encoding)
        ds.close()
        ds_sub.close()
        return ("ok", "{} KB".format(os.path.getsize(out_path) // 1024))
    except Exception as e:
        if os.path.exists(out_path):
            os.remove(out_path)
        return ("fail", str(e))
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


# Run downloads for both variants
variant_to_dir = {
    "clear_inst_v2": csif_clear_dir,
    "all_daily": csif_all_dir,
}

totals = {"ok": 0, "skip": 0, "fail": 0}
for variant_key, files in csif_inventory.items():
    out_dir = variant_to_dir[variant_key]
    print("\n[{}] {} files -> {}".format(variant_key, len(files), out_dir))
    for i, file_info in enumerate(files, 1):
        status, msg = download_and_subset(file_info, out_dir)
        totals[status] += 1
        if status == "fail":
            print("  [{}/{}] {} FAILED: {}".format(i, len(files), file_info["filename"], msg))
        elif i % 25 == 0 or i == len(files):
            print("  [{}/{}] processed ({} ok, {} skip, {} fail so far)".format(
                i, len(files), totals["ok"], totals["skip"], totals["fail"]))

print("\n" + "=" * 60)
print("CSIF Download + Subset Summary")
print("=" * 60)
print("  Downloaded + subset : {}".format(totals["ok"]))
print("  Skipped (existed)   : {}".format(totals["skip"]))
print("  Failed              : {}".format(totals["fail"]))

# Final size check
print("\nFinal subset storage:")
for variant_key, d in variant_to_dir.items():
    files = [f for f in os.listdir(d) if f.endswith(".nc")]
    total_mb = sum(os.path.getsize(os.path.join(d, f)) for f in files) / (1024 * 1024)
    print("  {}: {} files ({:.2f} MB)".format(variant_key, len(files), total_mb))
